### GitHub

[Video](https://youtu.be/TMIIEoHZkV8)

Dieses Notebook setzt das Notebook **Git** voraus.

[GitHub](https://github.com) ist ein Dienst, auf dem man Git-Repositorys im Internet ablegen kann. Wir nutzen ihn

- als Cloudspeicher für unsere lokalen Repos,
- um von verschiedenen PCs (zu Hause, in der Schule) am selben Projekt zu arbeiten,
- zur Zusammenarbeit mit anderen.

Ein Repository auf einem anderen Rechner heißt **Remote-Repository**. Git selbst braucht GitHub nicht: Jedes
Repository enthält die ganze Geschichte, GitHub ist nur ein gemeinsamer Treffpunkt. Ähnliche Dienste sind GitLab und
Codeberg.

#### Lokales Repo auf GitHub bringen

Auf GitHub legen wir (nach der Anmeldung) mit *New repository* ein leeres Repo *GitDemo* an und kopieren seine
Adresse. Dann verbinden wir unser lokales Repo damit und schicken den Branch *main* hinauf:

```
git remote add origin https://github.com/benutzer/GitDemo.git
git push -u origin main
```

- `origin` ist der übliche Name für das Remote-Repository, eine Abkürzung für die lange Adresse.
- `git push origin main` schickt den lokalen Branch *main* nach *origin*, dort heißt er auch *main*.
- `-u` (upstream) verknüpft den lokalen Branch mit dem auf GitHub. Danach reicht ein einfaches `git push` oder
  `git pull`.

Beim ersten `push` öffnet sich ein Browserfenster, in dem man sich bei GitHub anmeldet. Git für Windows merkt sich
die Anmeldung.

Mit anderen Branches geht es genauso, `git push --all origin` schickt alle Branches.

#### Push, Fetch und Pull

<img src='gh_bereiche.png' width='700'>

- `git push`: eigene Commits zum Remote-Repository hochladen
- `git fetch`: neue Commits vom Remote-Repository holen, ohne die eigenen Branches zu verändern
- `git pull`: `fetch` und danach `merge`, die neuen Commits kommen also auch in den eigenen Branch und ins
  Arbeitsverzeichnis

### Ausprobieren ohne GitHub

Für einen Versuch braucht man kein GitHub-Konto. Ein Remote-Repository kann auch ein Ordner auf demselben Rechner
sein. Wir legen in einem temporären Ordner drei Repositorys an:

- `github/GitDemo.git` spielt GitHub. Es ist ein *bare* Repository: nur die Geschichte, ohne Arbeitsverzeichnis.
- `schule` und `zuhause` sind die lokalen Repos auf zwei PCs.

Die Funktion `git(befehl, ort)` führt einen Git-Befehl im Ordner `ort` aus und zeigt alle Meldungen an.

In [1]:
import shlex
import subprocess
import tempfile
from pathlib import Path

basis = Path(tempfile.mkdtemp())

def git(befehl, ort):
    erg = subprocess.run(['git'] + shlex.split(befehl), cwd=basis / ort, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, encoding='utf-8', errors='replace')
    print(erg.stdout, end='')

def schreibe(ort, datei, text):
    (basis / ort / datei).write_text(text, encoding='utf-8')

def neues_repo(ort, name):
    (basis / ort).mkdir()
    git('init -q -b main', ort)
    git(f'config user.name "{name}"', ort)
    git('config user.email demo@example.com', ort)
    git('config pull.rebase false', ort)

(basis / 'github').mkdir()
git('init -q --bare -b main GitDemo.git', 'github')
neues_repo('schule', 'Schule')

In der Schule entstehen drei Commits, dann geht der Stand nach „GitHub“:

In [2]:
for i in range(1, 4):
    schreibe('schule', 'f1.txt', ''.join(f'{k}\n' for k in range(1, i + 1)))
    git('add .', 'schule')
    git(f'commit -q -m c{i}', 'schule')
git('remote add origin ../github/GitDemo.git', 'schule')
git('push -u origin main', 'schule')

branch 'main' set up to track 'origin/main'.
To ../github/GitDemo.git
 * [new branch]      main -> main


#### Klonen

Zu Hause gibt es das Repo noch nicht. `git clone` legt einen neuen Ordner an, holt die ganze Geschichte und stellt
`origin` gleich richtig ein. Bei GitHub lautet der Befehl z.B. `git clone https://github.com/benutzer/GitDemo.git`.
(Die zweite Zeile kürzt nur die Adresse, damit die Ausgaben übersichtlich bleiben.)

In [3]:
git('clone -q github/GitDemo.git zuhause', '.')
git('remote set-url origin ../github/GitDemo.git', 'zuhause')
git('config user.name Zuhause', 'zuhause')
git('config user.email demo@example.com', 'zuhause')
git('config pull.rebase false', 'zuhause')
git('log --oneline --decorate --all', 'zuhause')

34aa40c (HEAD -> main, origin/main, origin/HEAD) c3
048a285 c2
d3a1cf0 c1


`origin/main` ist ein **Remote-Tracking-Branch**: Er zeigt auf den Commit, auf dem *main* bei `origin` stand, als wir
zuletzt Verbindung hatten. Er wird nur durch `fetch`, `pull` und `push` bewegt.

#### Von zwei PCs aus arbeiten

<img src='gh_zwei_pcs.png' width='520'>

Zu Hause wird weitergearbeitet und hochgeladen:

In [4]:
schreibe('zuhause', 'f1.txt', '1\n2\n3\n4\n')
git('commit -q -am c4', 'zuhause')
git('push', 'zuhause')

To ../github/GitDemo.git
   34aa40c..d8df011  main -> main


In der Schule weiß Git davon noch nichts. `git fetch` holt die neuen Commits, *main* bleibt aber, wo er war:

In [5]:
git('fetch', 'schule')
git('status', 'schule')
git('log --oneline --decorate --all', 'schule')

From ../github/GitDemo
   34aa40c..d8df011  main       -> origin/main


On branch main
Your branch is behind 'origin/main' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)

nothing to commit, working tree clean
d8df011 (origin/main, origin/HEAD) c4
34aa40c (HEAD -> main) c3
048a285 c2
d3a1cf0 c1


<img src='gh_fetch.png' width='400'>

*Your branch is behind 'origin/main' by 1 commit*: Auf GitHub gibt es einen Commit, den der lokale Branch noch nicht
hat. `git pull` (bzw. hier `git merge`) holt ihn nach, das ist ein Fast-Forward:

In [6]:
git('pull', 'schule')
git('log --oneline --decorate --all', 'schule')

Updating 34aa40c..d8df011
Fast-forward
 f1.txt | 1 +
 1 file changed, 1 insertion(+)
d8df011 (HEAD -> main, origin/main, origin/HEAD) c4
34aa40c c3
048a285 c2
d3a1cf0 c1


#### Vergessen zu pullen

Wer vor der Arbeit vergisst zu pullen, bekommt beim Push eine Fehlermeldung. Zu Hause und in der Schule wird
gleichzeitig ein Commit gemacht, zu Hause wird zuerst gepusht:

In [7]:
schreibe('zuhause', 'f2.txt', 'von zu Hause\n')
git('add .', 'zuhause')
git('commit -q -m h1', 'zuhause')
git('push', 'zuhause')
schreibe('schule', 'f3.txt', 'aus der Schule\n')
git('add .', 'schule')
git('commit -q -m s1', 'schule')
git('push', 'schule')

To ../github/GitDemo.git
   d8df011..02d61c7  main -> main
To ../github/GitDemo.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to '../github/GitDemo.git'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


Der Push wird abgelehnt (*rejected*): Auf GitHub liegt ein Commit, den die Schule nicht hat. Würde Git annehmen, ginge
h1 verloren. Die Lösung steht im Hinweis: erst `git pull`, dann wieder `git push`. Weil beide Seiten neue Commits
haben, entsteht beim Pull ein Merge-Commit (bei Änderungen an derselben Stelle auch ein Konflikt, den man wie im
Notebook **Git** löst).

In [8]:
git('pull --no-edit', 'schule')
git('push', 'schule')
git('log --oneline --decorate --graph --all', 'schule')

From ../github/GitDemo
   d8df011..02d61c7  main       -> origin/main
Merge made by the 'ort' strategy.
 f2.txt | 1 +
 1 file changed, 1 insertion(+)
 create mode 100644 f2.txt


To ../github/GitDemo.git
   02d61c7..1668a22  main -> main
*   1668a22 (HEAD -> main, origin/main, origin/HEAD) Merge branch 'main' of ../github/GitDemo
|\  
| * 02d61c7 h1
* | 405f02c s1
|/  
* d8df011 c4
* 34aa40c c3
* 048a285 c2
* d3a1cf0 c1


Darum gilt beim Arbeiten an mehreren Rechnern: **vor der Arbeit `git pull`, nach der Arbeit `git push`.**

### Branches und Remote-Tracking

Auch andere Branches kann man hochladen. Mit `-u` wird der lokale Branch mit dem auf GitHub verknüpft, `git branch
-vv` zeigt die Verknüpfungen (*upstream*):

In [9]:
git('switch -q -c feature', 'schule')
schreibe('schule', 'f4.txt', 'neues Feature\n')
git('add .', 'schule')
git('commit -q -m d1', 'schule')
git('push -u origin feature', 'schule')
git('branch -vv', 'schule')

branch 'feature' set up to track 'origin/feature'.
To ../github/GitDemo.git
 * [new branch]      feature -> feature
* feature 2aaa0bb [origin/feature] d1
  main    1668a22 [origin/main] Merge branch 'main' of ../github/GitDemo


Ohne `-u` hätte der lokale Branch *feature* keinen upstream, ein einfaches `git push` würde mit einem Hinweis
abbrechen. Mit der Einstellung

```
git config --global push.autoSetupRemote true
```

richtet Git den upstream beim ersten `git push` eines neuen Branches selbst ein.

#### Clone und Branches

Beim Klonen holt Git alle Branches, legt aber lokal nur *main* an. Die anderen gibt es als Remote-Tracking-Branches.
`git branch -a` zeigt alle:

In [10]:
git('fetch -q', 'zuhause')
git('branch -a', 'zuhause')

* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/feature
  remotes/origin/main


`git switch feature` legt dann den lokalen Branch an und verknüpft ihn mit `origin/feature`:

In [11]:
git('switch feature', 'zuhause')
git('branch -vv', 'zuhause')

branch 'feature' set up to track 'origin/feature'.
Switched to a new branch 'feature'
* feature 2aaa0bb [origin/feature] d1
  main    02d61c7 [origin/main: behind 2] h1


<img src='gh_tracking.png' width='420'>

### Zusammenarbeit mit anderen

#### Collaborators

Zu einem eigenen Repository kann man auf GitHub Mitarbeitende einladen (*Settings, Collaborators*). Wer die Einladung
annimmt, hat Schreibrechte: Er kann pushen, Branches mergen und Pull Requests annehmen, aber z.B. keine weiteren
Mitarbeitenden einladen.

#### Pull Requests

Mit einem **Pull Request** schlägt man vor, die Änderungen eines Branches in einen anderen (meist *main*) zu
übernehmen. Auf GitHub sieht man dann die Änderungen, kann sie kommentieren, weitere Commits nachschieben und den Pull
Request schließlich annehmen (mergen) oder ablehnen. So schaut immer noch jemand auf den Code, bevor er in *main*
kommt.

#### Workflow 1: Feature-Branches

Zwei Collaborators arbeiten an einem gemeinsamen Repo. Jeder entwickelt sein Feature auf einem eigenen Branch
(*blau*, *gruen*) und pusht ihn. Für jeden Branch wird ein Pull Request gestellt, besprochen und angenommen. *main*
ändert sich nur durch angenommene Pull Requests.

<img src='gh_workflow.png' width='420'>

m1 und m2 sind die Merge-Commits der beiden angenommenen Pull Requests. Danach holen sich alle mit `git pull` den
neuen Stand von *main*.

#### Workflow 2: Fork und Pull Request

Bei einem fremden Open-Source-Projekt hat man keine Schreibrechte. Dann legt man auf GitHub mit *Fork* eine eigene
Kopie an, klont sie, arbeitet darin und pusht in den eigenen Fork. Von dort stellt man einen Pull Request an das
Original-Repo. Neue Commits des Originals holt man sich, indem man es als zweites Remote (meist `upstream` genannt)
einträgt und daraus pullt.

<img src='gh_fork.png' width='620'>

### Zusammenfassung

| Befehl | Wirkung |
|---|---|
| `git remote add origin adresse` | Remote-Repository unter dem Namen *origin* eintragen |
| `git push -u origin main` | Branch hochladen und verknüpfen, danach reicht `git push` |
| `git clone adresse` | Repository mit ganzer Geschichte herunterladen |
| `git fetch` | neue Commits holen, eigene Branches unverändert |
| `git pull` | `fetch` und `merge` |
| `git branch -vv`, `git branch -a` | Verknüpfungen und alle Branches anzeigen |

- Vor der Arbeit `git pull`, nach der Arbeit `git push`. Ein abgelehnter Push heißt: erst pullen.
- Zusammenarbeit über Feature-Branches und Pull Requests, bei fremden Projekten über einen Fork.